# Muharaf transformer: locked selection and held-out test

**Run this notebook once after the Stage 3 validation result.** The selected model is the completed second-epoch checkpoint because its validation CER (36.5503%) was lower than the first epoch's (37.9225%). The second epoch used a lower learning rate as well as more training. The mark-alignment diagnostic was inspected on validation before this lock.

This notebook records the selection and checkpoint fingerprint **before loading the test manifest**. It then evaluates the selected model and the original frozen ArTrOCR model on the same held-out test lines. It writes predictions as it progresses. No test result is used to change a model or select a different checkpoint.

Use a GPU runtime, ensure your Drive still contains the ZIP, set0 manifests, and prior run folders, then run the cells in order. The first cells stage the ZIP and check the manifests as before. If the runtime disconnects during test inference, rerun from the top; saved predictions resume under `final_test_v1`.


In [ ]:
%pip -q install -U "transformers>=4.50,<6" accelerate jiwer safetensors

In [ ]:
import os, sys, torch, transformers
from google.colab import drive
drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Change runtime type to GPU, then rerun.'
print('GPU:', torch.cuda.get_device_name(0))
print('Python:', sys.version.split()[0], 'PyTorch:', torch.__version__, 'Transformers:', transformers.__version__)

In [ ]:
from pathlib import Path
DATA_DIR = Path('/content/drive/MyDrive/Colab Notebooks/data-stuff')
assert DATA_DIR.is_dir(), f'Drive folder not found: {DATA_DIR}'
ZIP = DATA_DIR/'public_line_images.zip'
assert ZIP.is_file(), f'ZIP not found: {ZIP}'
candidate_dirs = [p.parent for p in DATA_DIR.rglob('validation.jsonl') if (p.parent/'train.jsonl').is_file()]
print('Archive:', ZIP)
print('Manifest directory candidates:', candidate_dirs)
assert len(candidate_dirs) == 1, 'Expected exactly one folder containing train.jsonl and validation.jsonl under data-stuff. Check the folder name/contents.'
MANIFESTS = candidate_dirs[0]
OUTPUT = DATA_DIR/'muharaf_transformer_runs'/'frozen_v1'
OUTPUT.mkdir(parents=True, exist_ok=True)
print('Outputs:', OUTPUT)

In [ ]:
import shutil, zipfile
LOCAL_ZIP = Path('/content/public_line_images.zip')
STAGING = Path('/content/muharaf_public')
if not LOCAL_ZIP.exists() or LOCAL_ZIP.stat().st_size != ZIP.stat().st_size:
    shutil.copyfile(ZIP, LOCAL_ZIP)
with zipfile.ZipFile(LOCAL_ZIP) as z:
    names = z.namelist()
    assert all(not Path(n).is_absolute() and '..' not in Path(n).parts for n in names), 'Archive contains unsafe names'
    assert any(n.startswith('public/') and n.endswith('.png') for n in names), 'Expected public/*.png inside ZIP'
    if not (STAGING/'public').exists():
        z.extractall(STAGING)
PUBLIC = STAGING/'public'
pngs = list(PUBLIC.rglob('*.png'))
print('Extracted PNGs:', len(pngs), 'at', PUBLIC)
assert len(pngs) == 24495, 'Expected 24,495 line images; check the ZIP and extraction.'
assert all(p.with_suffix('.txt').exists() for p in pngs), 'A line PNG is missing its TXT label.'

### Validate manifest fields and map lines to local files

The manifest may contain original paths from another computer. This cell maps by the PNG filename inside the ZIP, requires names to be unique, and checks that each manifest reference agrees with its unchanged TXT file. If your manifest uses different field names, the first printed record shows what to adapt.

In [ ]:
import json, collections
sample_record = json.loads((MANIFESTS/'train.jsonl').open(encoding='utf-8').readline())
print('Manifest field names:', list(sample_record))
print('One record (shortened):', {k:str(v)[:110] for k,v in sample_record.items()})

by_name = {}
for p in pngs:
    if p.name in by_name:
        raise ValueError(f'Duplicate image filename: {p.name}')
    by_name[p.name] = p

def pick(record, names):
    for k in names:
        if k in record and isinstance(record[k], str):
            return record[k]
    return None

def read_split(name):
    result = []
    with (MANIFESTS/f'{name}.jsonl').open(encoding='utf-8') as fh:
        for lineno, row in enumerate(fh, 1):
            r = json.loads(row)
            img = pick(r, ('image_path','image','image_file','png_path','line_image_path','line_image'))
            if not img:
                paths = [v for v in r.values() if isinstance(v,str) and v.lower().endswith('.png')]
                if len(paths) == 1: img = paths[0]
            assert img, f'No PNG field in {name}:{lineno}; inspect the printed record above'
            p = by_name.get(Path(img).name)
            assert p, f'Missing PNG for {name}:{lineno}: {img}'
            reference = p.with_suffix('.txt').read_text(encoding='utf-8')
            recorded = pick(r, ('text','transcription','ground_truth','gt','label_text','reference'))
            if recorded is not None:
                assert recorded == reference, f'Manifest text differs from original label for {p.name}'
            result.append({'image':p, 'reference':reference, 'source':r})
    assert len({x['image'] for x in result}) == len(result), f'Duplicate {name} image'
    return result

train, valid = read_split('train'), read_split('validation')
assert len(train) == 22092 and len(valid) == 1069, (len(train),len(valid))
assert not ({x['image'] for x in train} & {x['image'] for x in valid})
print('Training lines:',len(train),'Validation lines:',len(valid))
print('Example reference:',repr(valid[0]['reference'][:140]))

### Load the selected checkpoint and verify unchanged preprocessing


In [ ]:
from transformers import VisionEncoderDecoderModel, TrOCRProcessor
MODEL_ID = 'BushraAlmod03/ArTrOCR-HTR'
processor = TrOCRProcessor.from_pretrained(MODEL_ID)
SOURCE_CHECKPOINT = DATA_DIR/'muharaf_transformer_runs'/'finetune_v2_second_epoch'/'checkpoint_latest'
SOURCE_STATE_FILE = SOURCE_CHECKPOINT/'state.json'
assert SOURCE_STATE_FILE.is_file() and (SOURCE_CHECKPOINT/'model').is_dir(), 'Completed Stage 3 checkpoint missing.'
source_state = json.loads(SOURCE_STATE_FILE.read_text(encoding='utf-8'))
assert source_state['step'] == 1381 and source_state['micro_done'] == 11046, 'Stage 3 must have completed all 22,092 training lines.'
assert source_state['config']['model'] == MODEL_ID and source_state['config']['train_lines'] == len(train)
model = VisionEncoderDecoderModel.from_pretrained(SOURCE_CHECKPOINT/'model').to('cuda')
assert processor.image_processor.do_resize is False, 'Review processor resize behavior before proceeding.'
assert model.config.decoder_start_token_id is not None and model.config.pad_token_id is not None
special = processor.tokenizer.all_special_ids
bad = []
max_label_tokens = 0
for i, x in enumerate(train):
    ids = processor.tokenizer(x['reference'], add_special_tokens=False)['input_ids']
    max_label_tokens = max(max_label_tokens, len(ids))
    back = processor.tokenizer.decode(ids, skip_special_tokens=False, clean_up_tokenization_spaces=False)
    if back != x['reference'] or any(v in special for v in ids):
        bad.append((i,repr(x['reference'][:50]),repr(back[:50])))
print('Tokenizer non-roundtrip labels:',len(bad),'examples:',bad[:5])
assert not bad, 'Stop: tokenizer does not preserve every training label exactly.'
MAX_NEW_TOKENS = max_label_tokens + 8
decoder_limit = getattr(model.config.decoder, 'max_position_embeddings', None)
print('Longest training reference:',max_label_tokens,'tokens; generation allowance:',MAX_NEW_TOKENS,'decoder limit:',decoder_limit)
assert decoder_limit is None or MAX_NEW_TOKENS + 2 <= decoder_limit, 'A label exceeds the decoder position limit; review long lines before scoring.'

# The model card explicitly enables encoder position interpolation for generation.
if not hasattr(model.encoder, '_muharaf_original_forward'):
    model.encoder._muharaf_original_forward = model.encoder.forward
    def interpolated_forward(pixel_values, *args, **kwargs):
        kwargs['interpolate_pos_encoding'] = True
        return model.encoder._muharaf_original_forward(pixel_values, *args, **kwargs)
    model.encoder.forward = interpolated_forward
model.eval()
print('Selected epoch-2 weights loaded; single-line setup inference only.')

In [ ]:
from PIL import Image
import numpy as np
def prepare_image(path):
    with Image.open(path) as src:
        img = src.convert('RGB')
    w,h = img.size
    target_w,target_h = 512,102
    new_w = max(1,round(w * target_h / h))
    img = img.resize((new_w,target_h),Image.Resampling.LANCZOS)
    if new_w > target_w:
        img = img.resize((target_w,target_h),Image.Resampling.LANCZOS)
        new_w = target_w
    canvas = Image.new('RGB',(target_w,target_h),(255,255,255))
    canvas.paste(img,(0,0))
    return canvas

def predict(item):
    im = prepare_image(item['image'])
    pixels = processor(images=im,return_tensors='pt').pixel_values.to('cuda')
    with torch.inference_mode():
        # One decoder-start token plus the same generated-token allowance.
        # Passing max_length alone avoids the checkpoint's max_length=20 warning.
        ids = model.generate(pixels,max_length=MAX_NEW_TOKENS+1,num_beams=1)
    return processor.batch_decode(ids,skip_special_tokens=True,clean_up_tokenization_spaces=False)[0]

print('Example:',valid[0]['image'].name)
print('Reference:',repr(valid[0]['reference']))
print('Prediction:',repr(predict(valid[0])))

### Lock the selection before opening test
The fingerprint and validation criterion are recorded once. Rerunning checks that the lock is unchanged.


In [ ]:
import hashlib
from datetime import datetime, timezone

FINAL_OUTPUT = DATA_DIR/'muharaf_transformer_runs'/'final_test_v1'
FINAL_OUTPUT.mkdir(parents=True, exist_ok=True)
comparison_path = DATA_DIR/'muharaf_transformer_runs'/'finetune_v2_second_epoch'/'validation_comparison.json'
comparison = json.loads(comparison_path.read_text(encoding='utf-8'))
assert comparison['lines'] == 1069
assert abs(comparison['first_epoch']['raw_cer'] - 0.37922465208747513) < 1e-10
assert abs(comparison['second_epoch']['raw_cer'] - 0.36550340812269244) < 1e-10
assert comparison['second_epoch']['raw_cer'] < comparison['first_epoch']['raw_cer']

def fingerprint_model(path):
    files = sorted(p for p in path.rglob('*') if p.is_file())
    assert files
    h = hashlib.sha256()
    for p in files:
        h.update(str(p.relative_to(path)).encode('utf-8'))
        with p.open('rb') as stream:
            for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
                h.update(block)
    return h.hexdigest()

selection = {
    'selected': 'finetune_v2_second_epoch/checkpoint_latest',
    'criterion': 'lowest raw CER on set0 validation among the two completed fine-tuning epochs',
    'validation_cer_first': comparison['first_epoch']['raw_cer'],
    'validation_cer_second': comparison['second_epoch']['raw_cer'],
    'source_state': source_state,
    'model_sha256': fingerprint_model(SOURCE_CHECKPOINT/'model'),
    'max_new_tokens': MAX_NEW_TOKENS,
    'preprocessing': 'same 512x102 RGB preparation as Stage 1-3',
    'note': 'Validation mark alignment: first 206 matched / 385 unmatched predicted; second 229 / 413. Test unopened at selection.'
}
selection_path = FINAL_OUTPUT/'selection_lock.json'
if selection_path.exists():
    assert json.loads(selection_path.read_text(encoding='utf-8')) == selection, 'Selection lock differs; stop before test.'
else:
    selection_path.write_text(json.dumps(selection, ensure_ascii=False, indent=2), encoding='utf-8')
print('Selection locked:', selection['selected'])
print('Model SHA-256:', selection['model_sha256'])
print('Test manifest has not been loaded yet.')


### Load the held-out test and run each model once
Predictions append to separate files. If interrupted, completed lines are skipped on a rerun.


In [ ]:
test = read_split('test')
assert len(test) == 1334, f'Unexpected test line count: {len(test)}'
train_names = {x['image'].name for x in train}
valid_names = {x['image'].name for x in valid}
test_names = {x['image'].name for x in test}
assert not (test_names & train_names) and not (test_names & valid_names)
print('Held-out test lines:', len(test))


In [ ]:
def read_saved(path):
    saved = {}
    if path.exists():
        for line in path.open(encoding='utf-8'):
            row = json.loads(line)
            assert row['image_name'] not in saved, 'Duplicate prediction row.'
            saved[row['image_name']] = row
    return saved

def run_test_predictions(path):
    saved = read_saved(path)
    model.eval()
    with path.open('a', encoding='utf-8') as out:
        for idx, item in enumerate(test, 1):
            name = item['image'].name
            if name in saved:
                assert saved[name]['reference'] == item['reference']
                continue
            row = {'image_name': name, 'reference': item['reference'], 'prediction': predict(item)}
            out.write(json.dumps(row, ensure_ascii=False) + '\n')
            out.flush()
            saved[name] = row
            if idx % 25 == 0:
                print('Completed', idx, '/', len(test))
    assert set(saved) == {x['image'].name for x in test}
    return saved

selected_path = FINAL_OUTPUT/'selected_test_predictions.jsonl'
selected = run_test_predictions(selected_path)
print('Selected checkpoint test predictions complete:', len(selected))


In [ ]:
import gc
del model
gc.collect()
torch.cuda.empty_cache()
model = VisionEncoderDecoderModel.from_pretrained(MODEL_ID).to('cuda')
if not hasattr(model.encoder, '_muharaf_original_forward'):
    model.encoder._muharaf_original_forward = model.encoder.forward
    def interpolated_forward(pixel_values, *args, **kwargs):
        kwargs['interpolate_pos_encoding'] = True
        return model.encoder._muharaf_original_forward(pixel_values, *args, **kwargs)
    model.encoder.forward = interpolated_forward

frozen_path = FINAL_OUTPUT/'frozen_test_predictions.jsonl'
frozen_test = run_test_predictions(frozen_path)
print('Frozen checkpoint test predictions complete:', len(frozen_test))


### Test report
This comparison is descriptive. Do not use it to revise checkpoint selection.


In [ ]:
from jiwer import cer, wer
from rapidfuzz.distance import Levenshtein

names = [x['image'].name for x in test]
refs = [x['reference'] for x in test]
assert refs == [selected[n]['reference'] for n in names]
assert refs == [frozen_test[n]['reference'] for n in names]
marks = set('ًٌٍَُِّْ')
strip_marks = lambda s: ''.join(c for c in s if c not in marks)

def measure(preds):
    hyps = [preds[n]['prediction'] for n in names]
    matched = 0
    for ref, pred in zip(refs, hyps):
        for tag, r0, r1, _, _ in Levenshtein.opcodes(ref, pred):
            if tag == 'equal':
                matched += sum(c in marks for c in ref[r0:r1])
    ref_marks = sum(c in marks for s in refs for c in s)
    pred_marks = sum(c in marks for s in hyps for c in s)
    return {
        'raw_cer': cer(refs, hyps),
        'wer': wer(refs, hyps),
        'cer_without_basic_harakat': cer(list(map(strip_marks, refs)), list(map(strip_marks, hyps))),
        'reference_basic_harakat': ref_marks,
        'predicted_basic_harakat': pred_marks,
        'matched_basic_harakat': matched,
        'unmatched_reference_basic_harakat': ref_marks - matched,
        'unmatched_predicted_basic_harakat': pred_marks - matched
    }

result = {
    'model': MODEL_ID,
    'split': 'public_1100/set0 test',
    'lines': len(test),
    'selection': selection,
    'frozen': measure(frozen_test),
    'selected_second_epoch': measure(selected),
    'note': 'Mark alignment is a character-level diagnostic; visual presence is not certified.'
}
(FINAL_OUTPUT/'test_comparison.json').write_text(json.dumps(result, ensure_ascii=False, indent=2), encoding='utf-8')
print(json.dumps(result, ensure_ascii=False, indent=2))
print('Saved:', FINAL_OUTPUT)
